In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
import pyautogui
import time

chrome_options = webdriver.ChromeOptions()

# Desabilita proteção de downloads
chrome_options.add_experimental_option("prefs", {
    "safebrowsing.enabled": False,
    "safebrowsing.disable_download_protection": True,
    "download.prompt_for_download": False,
    "download.default_directory": "C:\\Users\\Suporte SBJ\\Downloads",
})

# Argumento extra para desabilitar o bloqueio de downloads não seguros
chrome_options.add_argument("--safebrowsing-disable-download-protection")
chrome_options.add_argument("--disable-features=InsecureDownloadWarnings")


In [2]:
driver = webdriver.Chrome(options=chrome_options)  # apenas UMA linha aqui
wait = WebDriverWait(driver, 20)

driver.get("http://107.21.0.176:8190/")
time.sleep(3)

In [3]:
# //*[@id="top"]/div/div[3]/section[1]/ul/li[2]/article/a/i
#botao_download = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="top"]/div/div[3]/section[1]/ul/li[2]/article/a/i')))
#botao_download.click()
#time.sleep(3)  # aguarda 5 segundos para a próxima etapa

In [4]:
import os
import requests
from urllib.parse import urljoin, urlparse

download_dir = r"C:\\Users\\Suporte SBJ\\Downloads"
os.makedirs(download_dir, exist_ok=True)

# acha o ícone e sobe pro <a>
icon = wait.until(EC.element_to_be_clickable(
    (By.XPATH, '//*[@id="top"]/div/div[3]/section[1]/ul/li[2]/article/a/i')
))
a = icon.find_element(By.XPATH, "./ancestor::a[1]")
href = a.get_attribute("href")
if not href:
    raise RuntimeError("O link de download não tem href (pode ser gerado por JavaScript).")

file_url = urljoin(driver.current_url, href)

# sessão requests com cookies do selenium
sess = requests.Session()
for c in driver.get_cookies():
    sess.cookies.set(c["name"], c["value"])

resp = sess.get(file_url, stream=True, timeout=60)
resp.raise_for_status()

# nome do arquivo (Content-Disposition ou fallback pela URL)
filename = None
cd = resp.headers.get("Content-Disposition", "")
if "filename=" in cd:
    filename = cd.split("filename=")[-1].strip().strip('"')

if not filename:
    path = urlparse(file_url).path
    filename = os.path.basename(path) or "download.bin"

dest = os.path.join(download_dir, filename)

with open(dest, "wb") as f:
    for chunk in resp.iter_content(1024 * 256):
        if chunk:
            f.write(chunk)

print("Download OK:", dest)

time.sleep(3) # aguarda 5 segundos antes do quit
driver.quit()


Download OK: C:\\Users\\Suporte SBJ\\Downloads\ibrokerimportadvancedwebstart.jsp
